In [1]:
print("Hello")

Hello


In [3]:
import os
from groq import Groq
import sqlite3
import streamlit as st
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core import documents
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

#import .env  # Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv() 


True

In [1]:
#import os
#from dotenv import load_dotenv
from pymongo import MongoClient
#from langchain_core.documents import Document

# 1. Load configuration from your .env file
#load_dotenv()

In [2]:

def get_mongo_data():
    """
    Connects to the local MongoDB and retrieves all documents from the 'employees' collection.
    """
    uri = os.getenv("MONGO_URI", "mongodb://localhost:28017/")
    db_name = os.getenv("MONGO_DB_NAME", "ckb_db")
    
    try:
        client = MongoClient(uri, serverSelectionTimeoutMS=5000)
        db = client[db_name]
        
        # We target your 'employees' collection
        collection = db["employees"]
        
        # Fetch all documents
        docs = list(collection.find({}))
        print(f"✅ Successfully fetched {len(docs)} documents from {db_name}.employees")
        return docs

    except Exception as e:
        print(f"❌ Error connecting to MongoDB: {e}")
        return []


In [4]:

def convert_to_langchain_documents(mongo_docs):
    """
    Converts MongoDB dictionaries into LangChain Document objects.
    This makes them ready for Vector Stores like ChromaDB.
    """
    langchain_docs = []
    
    for doc in mongo_docs:
        # 1. Handle the MongoDB '_id' (it's an ObjectId, which must be a string for LangChain)
        doc_id = str(doc.pop("_id", "unknown_id"))
        
        # 2. Create the 'page_content' 
        # This is the text the LLM will actually read and search through.
        # We convert the dictionary to a clean string format.
        content = f"Employee Record Details: {doc}"
        
        # 3. Create the Document object with Metadata
        # Metadata is useful for filtering searches later (e.g., search only 'employees')
        lc_doc = Document(
            page_content=content,
            metadata={
                "source": "local_mongodb",
                "mongo_id": doc_id,
                "collection": "employees"
            }
        )
        langchain_docs.append(lc_doc)
        
    return langchain_docs


In [ ]:

    
def create_vector_store(langchain_docs):
    # 1. Initialize the embedding model
    # MiniLM-L6-v2 is a great choice—fast and efficient for local dev
    embedding_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

    # 2. Define your path (on D: drive as you requested)
    persist_dir = r'D:/GitHub/ckb_ai/testing/chroma_db'
    
    # 3. Create and store the vectors
    # from_documents handles the embedding and saving automatically
    vector_store = Chroma.from_documents(
        documents=langchain_docs,
        embedding=embedding_model,
        persist_directory=persist_dir
    )

    # Note: NO NEED for vector_store.persist() here anymore!
    
    print(f'✅ {len(langchain_docs)} Embeddings stored in ChromaDB at: {persist_dir}')
    return vector_store

In [9]:
client = Groq(api_key=os.environ.get('GROQ_API_KEY'))
#retriever = vector_store.as_retriever(search_kwargs={'k': 2})
def ask_groq(context, question):
    prompt = f"""
    Use the following context to answer the question.

    Context:
    {context}

    Question:
    {question}
    """

    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role': 'user', 'content': prompt}]
    )

    return response.choices[0].message.content


In [ ]:
if __name__ == "__main__":
    data = get_mongo_data()
    doc = convert_to_langchain_documents(data)
    vector_store = create_vector_store(doc)
    retriever = vector_store.as_retriever(search_kwargs={'k': 2})
    


✅ Successfully fetched 8 documents from ckb_db.employees


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 8 Embeddings stored in ChromaDB at: D:/GitHub/ckb_ai/testing/chroma_db

Answer:

The contact details of Krishna Shinde are:

Email: krishna@company.com 

Note: There is no other contact information (like phone number or address) provided in the given context.


In [15]:
query = input('Ask your question: ')
results = retriever.invoke(query)
context = '\n'.join([doc.page_content for doc in results])
answer = ask_groq(context, query)
print('\nAnswer:\n')
print(answer)



Answer:

Based on the provided context, there are 2 employees who work in the Sales department. Here are their details:

1. **Krishna Shinde**
   - Department: Sales
   - Email: krishna@company.com
   - Salary: 80000
   - Description: It is the senior sales manager and handles the sales team.

2. **Rahul Mehta**
   - Department: Sales
   - Email: rahul@company.com
   - Salary: 75000
   - Description: Handles enterprise sales and client acquisition.

Therefore, the total number of employees working in the Sales department is **2**.
